# Solution 06 — the joker and the critic

Two named agents, three runs each, a custom span inside the joker's tool. After running,
open the Logfire **Agents view**: both agents appear with run counts and cost; the **Live
view** shows the custom span nested in joker's traces.

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

In [ ]:
from pydantic_ai import Agent, RunContext

joker = Agent(
    MODEL,
    name="joker",
    instructions="You tell one short, clean programming joke. One-liners preferred.",
)
critic = Agent(
    MODEL,
    name="critic",
    instructions="Rate any joke from 0-10 and justify the score in one sentence.",
)

SETUP_LINES = {"punchline_count": 0}


@joker.tool
async def get_setup_lines(ctx: RunContext[None]) -> list[str]:
    """Fetch classic joke setup lines the joke should riff on."""
    with logfire.span("fetch_setup_lines"):  # ← your custom span in the trace
        SETUP_LINES["punchline_count"] += 1
        return ["A bug walks into a bar...", "My code was working yesterday..."]


for prompt in ["Tell me a joke.", "Another one — database themed!", "One more, about CSS."]:
    r = await joker.run(prompt)
    print("JOKE  :", r.output)

for prompt in [
    'Rate: "There are two hard things in CS: cache invalidation, naming, and off-by-ones."',
    'Rate: "I would tell you a UDP joke, but you might not get it."',
    'Rate: "The best thing about a boolean is that even if you are wrong, you are only off by a bit."',
]:
    r = await critic.run(prompt)
    print("CRITIC:", r.output)

**Notes**

- In the **Agents view** the two rows (`joker`, `critic`) now have run counts, cost,
  and duration stats — naming agents is what makes the dashboard useful.
- Drill into any joker trace and the `fetch_setup_lines` span sits inside the tool-call
  span, exactly where you opened it.
- Cost per agent over many runs is the number that later justifies switching a model.